<a href="https://colab.research.google.com/github/galipaydin/pes3850/blob/main/notebooks/lab09_pose_estimation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# PES 3850 · Lab 9 — Pose estimation on your own video
**Artificial Intelligence Applications in Sport · Week 10**

⏱️ About 90 minutes · 🧪 This notebook creates its own data — nothing can fail to download · 🎥 bring the squat and CMJ clips you filmed as homework

> **First:** `File → Save a copy in Drive`. Cells marked ✏️ **EDIT ME** are the ones you change.


### What you will produce
A knee-angle time series from video, a peak-flexion measurement, and — the point of the lab — **the difference
between what the AI measured and what you measured by hand in Kinovea.**

### Why the validation half is not optional
Markerless pose estimation is genuinely useful and genuinely limited. It is good at *within-athlete change under
fixed camera conditions*. It is much weaker at absolute joint angles, at high velocity, out of the sagittal plane,
and under occlusion. The only way to know which situation you are in is to measure against something you trust.

> **Two paths.** Block 3a runs MediaPipe on your uploaded clip. Block 3b loads a pre-computed landmark track so
> the analysis and the graded interpretation work even if the upload fails or the runtime dies.


---
## Block 1 — Setup

MediaPipe is only needed for Block 3a. Everything after Block 3 is plain numpy and works either way.


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
RNG = np.random.default_rng(10)
FPS = 60

# MediaPipe is only needed for Block 3a (the real path). Everything after Block 3
# is plain numpy and runs either way.
# Uncomment for the real pose-estimation path (about 40 s):
# !pip install -q mediapipe
print('Setup complete.')


---
## Block 2 — Filming rules

| Rule | Why |
|---|---|
| Sagittal view, camera perpendicular to the plane of motion | Every degree of camera yaw becomes an error in your knee angle — you will measure this in Block 6 |
| Phone on a fixed support, not held | A moving camera moves every landmark |
| Whole body in frame, feet included | The model estimates a hidden ankle badly |
| 60 fps if your phone offers it | At 30 fps a countermovement jump is about 20 frames of flight |
| ≤ 15 seconds, ≤ 720p | Longer or larger and the free Colab runtime will not finish |
| Plain background, contrasting clothing | Occlusion and camouflage are the two commonest failures |


---
## Block 3a — Run MediaPipe on your clip (the real path)

Uncomment, upload your squat clip, and run. It downloads the model once, then finds 33 landmarks in
every frame.

> **If you have seen MediaPipe code elsewhere, it probably looked different.** The old `mp.solutions.pose`
> interface is the legacy API — unsupported since 2023 and now being removed. This uses the current
> Tasks API, which is what the Week 1 demo used too.


In [ ]:
# ─── Uncomment this whole cell for the real path ───
#
# import cv2, mediapipe as mp
# from mediapipe.tasks import python as mp_python
# from mediapipe.tasks.python import vision
# from pathlib import Path
# from google.colab import files
#
# # 1. the model file (downloaded once)
# MODEL = 'pose_landmarker_full.task'
# if not Path(MODEL).exists():
#     !wget -q -O {MODEL} https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/latest/pose_landmarker_full.task
#
# # 2. your clip
# up  = files.upload()
# src = list(up.keys())[0]
#
# # 3. run it, in VIDEO mode so the model tracks between frames
# options = vision.PoseLandmarkerOptions(
#     base_options=mp_python.BaseOptions(model_asset_path=MODEL),
#     running_mode=vision.RunningMode.VIDEO,
#     num_poses=1)
#
# rows = []
# with vision.PoseLandmarker.create_from_options(options) as landmarker:
#     cap = cv2.VideoCapture(src)
#     fps = cap.get(cv2.CAP_PROP_FPS) or 30
#     i = 0
#     while True:
#         ok, frame = cap.read()
#         if not ok: break
#         mp_img = mp.Image(image_format=mp.ImageFormat.SRGB,
#                           data=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
#         res = landmarker.detect_for_video(mp_img, int(i / fps * 1000))
#         if res.pose_landmarks:
#             lm = res.pose_landmarks[0]
#             # indices: 24 right hip · 26 right knee · 28 right ankle
#             rows.append(dict(t=i/fps,
#                              hip_x=lm[24].x,   hip_y=lm[24].y,
#                              knee_x=lm[26].x,  knee_y=lm[26].y,
#                              ankle_x=lm[28].x, ankle_y=lm[28].y,
#                              vis=min(lm[24].visibility, lm[26].visibility, lm[28].visibility)))
#         i += 1
#     cap.release()
#
# track = pd.DataFrame(rows)
# FPS = int(fps)
# print(f'{i} frames, landmarks in {len(track)} ({100*len(track)/i:.0f}%)')
# print('Any frame with vis < 0.5 is a guess — do not quote an angle from it.')


---
## Block 3b — Pre-computed track (fallback / demonstration)

A right-side hip, knee and ankle through three squats and one countermovement jump, in normalised image
coordinates — the same shape MediaPipe returns. Run this if you did not use Block 3a.


In [ ]:
def limb_track(kind='squat', reps=3, fps=FPS):
    """Sagittal hip/knee/ankle positions for a squat or a countermovement jump.

    Returned in normalised image coordinates (0-1), with y increasing downwards —
    the same convention MediaPipe uses.
    """
    THIGH, SHANK = 0.42, 0.40         # segment lengths in normalised image units
    ANKLE_XY     = np.array([0.50, 0.92])
    M_PER_UNIT   = 1.80               # a full frame height is about 1.8 m of real space

    if kind == 'squat':
        dur, depth = 3.0, 105.0
        t   = np.arange(int(dur*reps*fps)) / fps
        ph  = (t % dur) / dur
        flex = 8 + (depth-8) * np.sin(np.pi*np.clip(ph/0.85, 0, 1))**1.6
        rise = np.zeros_like(t)
    else:
        # a real countermovement jump, phase by phase
        FLIGHT_S = 0.50               # 0.50 s of flight  ->  h = g*t^2/8 = 30.7 cm
        dur = 1.90
        t   = np.arange(int(dur*reps*fps)) / fps
        ph  = t % dur
        flex, rise = np.full_like(t, 8.0), np.zeros_like(t)
        for i, p in enumerate(ph):
            if p < 0.45:                                   # quiet stance
                flex[i] = 8
            elif p < 0.75:                                 # countermovement down
                flex[i] = 8 + 80 * (p-0.45)/0.30
            elif p < 0.95:                                 # propulsion
                flex[i] = 88 - 68 * (p-0.75)/0.20
            elif p < 0.95 + FLIGHT_S:                      # FLIGHT — true projectile
                tf = p - 0.95
                v0 = 9.81 * FLIGHT_S / 2                   # take-off velocity, m/s
                flex[i] = 20
                rise[i] = (v0*tf - 0.5*9.81*tf**2) / M_PER_UNIT
            elif p < 0.95 + FLIGHT_S + 0.25:               # landing absorption
                flex[i] = 20 + 55 * (p - 0.95 - FLIGHT_S)/0.25
            else:
                flex[i] = 75 - 67 * (p - 1.20 - FLIGHT_S)/(dur - 1.20 - FLIGHT_S)

    n = len(t)
    knee_ang = np.clip(flex, 0, 150) + RNG.normal(0, 0.9, n)      # tracking jitter
    th = np.radians(knee_ang)
    ankle = np.tile(ANKLE_XY, (n, 1)); ankle[:, 1] -= rise        # y grows downwards
    knee  = ankle + np.c_[np.zeros(n), -SHANK*np.ones(n)]
    hip   = knee  + np.c_[THIGH*np.sin(th), -THIGH*np.cos(th)]
    jit = lambda a: a + RNG.normal(0, .0016, a.shape)
    ankle, knee, hip = jit(ankle), jit(knee), jit(hip)
    return pd.DataFrame({'t': t,
        'hip_x': hip[:,0], 'hip_y': hip[:,1], 'knee_x': knee[:,0], 'knee_y': knee[:,1],
        'ankle_x': ankle[:,0], 'ankle_y': ankle[:,1]})

track      = limb_track('squat', reps=3)
track_jump = limb_track('cmj',   reps=2)
print(f'squat: {len(track)} frames · CMJ: {len(track_jump)} frames · {FPS} fps')
track.head()


---
## Block 4 — From landmarks to a joint angle

The knee angle is the angle at the knee between the thigh vector (knee→hip) and the shank vector (knee→ankle).
Reported here as **flexion**: 0° is a straight leg.


In [ ]:
def knee_flexion(df):
    thigh = np.c_[df.hip_x  - df.knee_x, df.hip_y  - df.knee_y]
    shank = np.c_[df.ankle_x- df.knee_x, df.ankle_y- df.knee_y]
    cos = (thigh*shank).sum(1) / (np.linalg.norm(thigh,axis=1)*np.linalg.norm(shank,axis=1))
    return 180 - np.degrees(np.arccos(np.clip(cos, -1, 1)))

track['knee_flex_deg'] = knee_flexion(track)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(track.t, track.knee_flex_deg, color='#8A1C2B', lw=1.6)
ax.set_xlabel('Time (s)'); ax.set_ylabel('Knee flexion (°)')
ax.set_title('Three bodyweight squats — knee flexion from video')
ax.grid(alpha=.3); plt.tight_layout(); plt.show()

print(f'Peak knee flexion : {track.knee_flex_deg.max():.1f}°')
print(f'Range of motion   : {track.knee_flex_deg.max()-track.knee_flex_deg.min():.1f}°')


---
## Block 5 — Phases, and a jump height

Eccentric = going down (flexion increasing). Concentric = coming up. For the CMJ we estimate flight time from
the hip trajectory and convert it to jump height with the standard flight-time formula, *h = g·t²/8*.


In [ ]:
# --- squat phases ---
dd = np.gradient(track.knee_flex_deg.rolling(5, center=True, min_periods=1).mean())
phase = np.where(dd > .25, 'eccentric', np.where(dd < -.25, 'concentric', 'transition'))

fig, ax = plt.subplots(2, 1, figsize=(10, 6.4))
cols = {'eccentric':'#8A1C2B','concentric':'#1B6470','transition':'#CFC6BF'}
for pnm, col in cols.items():
    m = phase == pnm
    ax[0].scatter(track.t[m], track.knee_flex_deg[m], s=6, color=col, label=pnm)
ax[0].legend(frameon=False, ncol=3); ax[0].set_ylabel('Knee flexion (°)')
ax[0].set_title('Squat phases'); ax[0].grid(alpha=.3)

# --- CMJ flight, detected from the ankle: the foot leaves the ground ---
track_jump['knee_flex_deg'] = knee_flexion(track_jump)
ank = track_jump.ankle_y.values
ground = np.percentile(ank, 95)
airborne = ank < ground - 0.008          # ✏️ EDIT ME — the take-off threshold

# measure each contiguous flight separately
runs, cur = [], 0
for a in airborne:
    if a: cur += 1
    elif cur: runs.append(cur); cur = 0
if cur: runs.append(cur)
runs = [r for r in runs if r > 3]

flight_time = np.mean(runs) / FPS
jump_h_cm   = 9.81 * flight_time**2 / 8 * 100

ax[1].plot(track_jump.t, -ank, color='#9A5714', lw=1.6)
ax[1].fill_between(track_jump.t, -ank, -ground, where=airborne, alpha=.25, color='#9A5714')
ax[1].set_xlabel('Time (s)'); ax[1].set_ylabel('Ankle height (image units)')
ax[1].set_title('Countermovement jump — shaded region is flight'); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

print(f'Jumps detected : {len(runs)}  (flights of {runs} frames)')
print(f'Flight time    : {flight_time*1000:.0f} ms')
print(f'Jump height    : {jump_h_cm:.1f} cm   (h = g·t²/8)')

one_frame_ms = 1000/FPS
h_plus = 9.81 * (flight_time + 1/FPS)**2 / 8 * 100
print(f'\nAt {FPS} fps one frame is {one_frame_ms:.1f} ms. Miss the take-off or the landing frame by one')
print(f'and the jump height moves by {h_plus - jump_h_cm:.1f} cm — before any question of pose accuracy.')
print(f'At 30 fps that error doubles. This is why frame rate, not the AI, is often your limiting factor.')


---
## Block 6 — Break it 💥 Camera angle

Everything above assumed the camera was perpendicular to the plane of motion. It never quite is.

Here the *same* squat is filmed from four camera yaw angles. The movement is identical; only the camera moved.


In [ ]:
def project(df, yaw_deg):
    """Re-project the sagittal-plane motion as seen by a camera rotated by yaw_deg."""
    k = np.cos(np.radians(yaw_deg))          # horizontal foreshortening
    out = df.copy()
    for j in ['hip','knee','ankle']:
        out[f'{j}_x'] = 0.5 + (df[f'{j}_x'] - 0.5) * k
    return out

res = []
for yaw in [0, 10, 20, 30, 45]:
    p = project(track, yaw)
    res.append({'camera yaw (°)': yaw,
                'peak knee flexion (°)': round(knee_flexion(p).max(), 1)})
res = pd.DataFrame(res)
res['error (°)'] = (res['peak knee flexion (°)'] - res.loc[0,'peak knee flexion (°)']).round(1)

fig, ax = plt.subplots(figsize=(7.5, 3.6))
ax.plot(res['camera yaw (°)'], res['peak knee flexion (°)'], 'o-', color='#8A1C2B', lw=2)
ax.axhline(res.loc[0,'peak knee flexion (°)'], ls='--', color='#999', label='true value')
ax.set_xlabel('Camera yaw away from perpendicular (°)')
ax.set_ylabel('Measured peak knee flexion (°)')
ax.set_title('The movement did not change. The camera did.')
ax.legend(frameon=False); ax.grid(alpha=.3); plt.tight_layout(); plt.show()

print(res.to_string(index=False))


**Read the numbers, not the intuition.** Yaw is more forgiving than most people expect at small angles — the
foreshortening goes with the cosine, so 10° costs a fraction of a degree and 20° costs about one. Past 30° it
climbs quickly.

Two things follow, and they point in opposite directions:

1. **Do not panic about a slightly imperfect camera.** For a sagittal squat, a careful phone setup is adequate.
2. **Do panic about anything that is not in the filmed plane.** This demonstration only rotated the camera; it did
   not move the knee. A 2D camera cannot see knee valgus, hip rotation or trunk lean out of plane *at all* — it
   projects them onto the image and reports a sagittal number as though nothing happened. That failure is silent,
   and it is far larger than the yaw error you just measured.

This is why honest use of single-camera markerless capture means **within-athlete change, in the filmed plane,
under fixed camera conditions** — and why an absolute joint angle from an uncalibrated phone should not go into
a medical record.


---
## Block 7 — The validation: MediaPipe vs Kinovea ✍️

Open your clip in **Kinovea**, find the frame of deepest squat, and measure the knee angle by hand with the angle
tool. Enter both numbers below. The class pools its results into one plot.


In [ ]:
# ✏️ EDIT ME — your two measurements
MY_NAME            = 'your name'
AUTOMATIC_PEAK_DEG = float(track.knee_flex_deg.max())   # or type your own MediaPipe result
KINOVEA_PEAK_DEG   = 103.0                              # ✏️ what you measured by hand

diff = AUTOMATIC_PEAK_DEG - KINOVEA_PEAK_DEG
print(f'MediaPipe : {AUTOMATIC_PEAK_DEG:.1f}°')
print(f'Kinovea   : {KINOVEA_PEAK_DEG:.1f}°')
print(f'Difference: {diff:+.1f}°')

# the class's pooled results — the instructor will paste the real numbers here in the session
cls = pd.DataFrame({'student': [f'P{i:02d}' for i in range(20)],
                    'difference_deg': RNG.normal(2.1, 4.4, 20).round(1)})
cls.loc[len(cls)] = [MY_NAME, round(diff,1)]

fig, ax = plt.subplots(figsize=(9, 3.4))
ax.axhline(0, color='#999')
ax.bar(cls.student, cls.difference_deg,
       color=['#8A1C2B' if s == MY_NAME else '#1B6470' for s in cls.student])
ax.set_ylabel('MediaPipe − Kinovea (°)'); ax.set_xlabel('Student')
ax.set_title('The class as its own validity study')
plt.xticks(rotation=60, ha='right', fontsize=8); ax.grid(axis='y', alpha=.3)
plt.tight_layout(); plt.show()

print(f'Class bias (mean difference) : {cls.difference_deg.mean():+.1f}°')
print(f'Limits of agreement (±1.96 SD): {cls.difference_deg.mean()-1.96*cls.difference_deg.std():+.1f}° '
      f'to {cls.difference_deg.mean()+1.96*cls.difference_deg.std():+.1f}°')


---
## Interpretation ✍️ **(graded)**

About 200 words:

1. State your MediaPipe−Kinovea difference and the class limits of agreement.
2. **Would you use this method to track one athlete's squat depth across a season?** Justify with your number.
3. **Would you use it to compare two different athletes?** Justify with your number. The answers should differ.
4. Your clip was filmed at some unknown camera yaw. Using Block 6, put a bound on how much of your difference
   could be camera placement rather than the algorithm.
5. A company sells markerless capture as “lab-grade accuracy”. What do you ask them?

---

**Your answer:**

*(replace this line)*


---
## AI Use Log

Complete honestly — it is part of the mark. Write *none* if you used no AI tool.

| Tool | What I asked it | What it gave me | What I changed or rejected |
|---|---|---|---|
|  |  |  |  |


---
### Submitting

1. `File → Save a copy in Drive` · 2. `Share → Anyone with the link → Viewer` · 3. Paste the link into ODTÜClass.
